# Steps
1. Convert token và lemma raw text to csv to get the index
2. Create a list of lemma_POS
3. Extract sentences with lemma_POS in token and lemma csv into individual files for each lemma_POS
4. Parse lại token bằng Stanza
5. Check lại giữa tag cũ và mới xem tỉ lệ sai POS là bao nhiêu

## Import

In [ ]:
import pandas as pd
import os
import stanza
import re
import sys
from pathlib import Path
import shutil
from tqdm import tqdm

sys.path.append('../data_preprocessing')
from utils import open_txt, save_to_txt, search_in_txt, replace_in_txt, return_stanza_parsed_tags

In [ ]:
def convert_org_stanza(org_lemma_pos):
    org_lemma = org_lemma_pos.rsplit('_')[0]
    org_pos = org_lemma_pos.split('_')[-1]

    stanza_lemma = org_lemma
    if org_pos == 'N':
        stanza_pos = 'NOUN'
    elif org_pos == 'V':
        stanza_pos = 'VERB'
    elif org_pos == 'A':
        stanza_pos = 'ADJ'
    
    return stanza_lemma, stanza_pos

In [ ]:
pilot_folder = Path(f'./SemEval_lat/')
pilot_folder_SemEval = Path(f'./SemEval_lat_SemEval/')

## Convert token and lemma raw text to csv to get the index

In [ ]:
corp_nos = [1, 2]
data_types = ['token', 'lemma']

In [ ]:
for corp_no in corp_nos:
    for data_type in data_types:
        raw_file = f'./semeval2020_ulscd_lat/corpus{corp_no}/{data_type}/lat{corp_no}.txt'
        raw_csv_path = f'./semeval2020_ulscd_lat/corpus{corp_no}/{data_type}/lat{corp_no}.csv'

        # Read txt
        with open(raw_file, 'r') as f:
            lines = f.readlines()
            lines = [line.rstrip('\n') for line in lines]
            # lines = [line.strip() for line in lines if line.strip()]  
            # Save to DataFrame and then to CSV
            df = pd.DataFrame(lines)
            df.columns = ['sent']
            df.to_csv(raw_csv_path, index=False, header=True)

In [ ]:
raw_file = './semeval2020_ulscd_lat/corpus1/lemma/lat1.txt'
with open(raw_file, 'r') as f:
    raw_lines = f.readlines()

print("Total lines in txt:", len(raw_lines))
print("Non-empty lines:", sum(1 for l in raw_lines if l.strip()))


## Create a list of lemma_POSs

In [ ]:
selected_lemmas = [
    'acerbus_A',
    'adsumo_V',
    'ancilla_N',
    'beatus_A',
    'civitas_N',
    'cohors_N',
    'consilium_N',
    'consul_N',
    'credo_V',
    'dolus_N',
    'dubius_A',
    'dux_N',
    'fidelis_A',
    'honor_N',
    'hostis_N',
    'humanitas_N',
    'imperator_N',
    'itero_V',
    'jus_N',
    'licet_V',
    'necessarius_A',
    'nepos_N',
    'nobilitas_N',
    'oportet_V',
    'poena_N',
    'pontifex_N',
    'potestas_N',
    'regnum_N',
    'sacramentum_N',
    'salus_N',
    'sanctus_A',
    'sapientia_N',
    'scriptura_N',
    'senatus_N',
    'sensus_N',
    'simplex_A',
    'templum_N',
    'titulus_N',
    'virtus_N',
    'voluntas_N'
]

## Re-parse with Stanza

In [ ]:
out_folder_reparsed = f'./SemEval_lat/corpus{corp_no}/reparsed/'
os.makedirs(os.path.dirname(out_folder_reparsed), exist_ok=True)

In [ ]:
stanza.download('la')
nlp = stanza.Pipeline(
        'la',
        processors='tokenize,mwt,pos,lemma,depparse',
        use_gpu=True,
        verbose=False,
        tokenize_no_ssplit=True
    )

In [ ]:
for corp_no in corp_nos:
    token_df = pd.read_csv(f'./SemEval_lat/corpus{corp_no}/token/lat{corp_no}.csv')

    reparsed_sents = []  # mỗi phần tử = 1 câu ở dạng nhiều dòng

    i = 0
    for sent in tqdm(token_df['sent']):
        doc = nlp(sent)
        for s in doc.sentences:
            lines = []
            lines.append(f'<s id=lat{corp_no}_{i}>')
            for w in s.words:
                lines.append(
                    f"{w.text}\t{w.lemma}\t{w.upos}\t{w.id}\t{w.head}\t{w.deprel}"
                )
            lines.append("</s>")
            reparsed_sents.append("\n".join(lines))
            i += 1

    # Save reparsed sentences to file
    with open(f'{out_folder_reparsed}/ccoha{corp_no}_reparsed.txt', 'w') as f:
        f.write("\n\n".join(reparsed_sents))

In [ ]:
# Lowercase the lemma form in the reparsed files
for corp_no in [1, 2]:
    reparsed_file = f'./SemEval_lat/corpus{corp_no}/reparsed/lat{corp_no}_reparsed.txt'

    with open(reparsed_file, 'r') as f:
        content = f.read()

    lines = content.split('\n')
    normalised_lines = []
    for line in lines:
        if line.startswith('<s id=') or line.startswith('</s>') or line.strip() == '':
            normalised_lines.append(line)
        else:
            parts = line.split('\t')
            if len(parts) >= 2:
                parts[1] = parts[1].lower()
                normalised_lines.append('\t'.join(parts))
            else:
                normalised_lines.append(line)

    with open(reparsed_file, 'w') as f:
        f.write('\n'.join(normalised_lines))

## Check, quantify the mismatches

In [ ]:
# MAIN FUNCTION TO CHECK FOR MISMACHES

# Create a dictionary to store mismatch reports for selected_lemmas
report_mismatches = {}

for corp_no in [1, 2]:
    report_mismatches[corp_no] = {}

    # Org lemma file
    org_parsed_file = f'./SemEval_lat/corpus{corp_no}/lemma/lat{corp_no}.csv'
    org_df = pd.read_csv(org_parsed_file)

    # Reparsed file
    reparsed_file = f'./SemEval_lat/corpus{corp_no}/reparsed/lat{corp_no}_reparsed.txt'
    with open(reparsed_file, 'r') as f:
        reparsed_content = f.read()
        
        reparsed_sents = reparsed_content.strip().split("\n\n")
        reparsed_sent_count = len(reparsed_sents)

    # Check no of sents
    org_df['sent'] = org_df['sent'].fillna('') # Fillna because there are some empty lines
    org_sents = org_df['sent'].tolist() 
    org_sent_count = len(org_sents)

    if org_sent_count != reparsed_sent_count:
        report_mismatches[corp_no]['Mismatched numbers of sentences'] = f"org={org_sent_count}, reparsed={reparsed_sent_count}"
        continue
    
    # Statistics for each lemma
    for selected_lemma in selected_lemmas:
        selected_lemma_base = selected_lemma.rsplit('_', 1)[0] # In other datasets than English, there is no _POS in the target

        # Whole file statistics:
        mismatch_sent = 0
        org_miss_lemma_count = 0
        reparsed_miss_lemma_count = 0
        report_mismatches[corp_no][selected_lemma] = []
        
        # Count selected_lemma_base in file
        pattern = rf'\b{re.escape(selected_lemma_base)}\b'

        org_lemma_count = org_df['sent'].str.count(pattern).sum()
        if org_lemma_count == 0: # Avoid division by zero error if the lemma is not found
            org_lemma_count = 1
        
        # Individual sentence check
        for i in range(org_sent_count):
            org_sent = org_sents[i]
            reparsed_sent = reparsed_sents[i]

            # Count selected_lemma_base occurrences in original sentence
            org_count = len(re.findall(pattern, org_sent))

            # Count selected_lemma occurrences in reparsed sentence
            stanza_lemma, stanza_pos = convert_org_stanza(selected_lemma)
            stanza_format = f'\t{stanza_lemma}\t{stanza_pos}\t'
            reparsed_count = reparsed_sent.count(stanza_format)

            # Return stanza tags for the selected_lemma
            stanza_tags = return_stanza_parsed_tags(reparsed_sent, selected_lemma)

            if org_count != reparsed_count:
                mismatch_sent += 1
                (report_mismatches[corp_no][selected_lemma].append(
                    f"Mismatch sentence:{i}, "
                    f"org={org_count}, "
                    f"reparsed={reparsed_count}, "
                    f"org_sent='{org_sent}, "
                    f"stanza_pos={stanza_tags}"))
                
                if org_count >= reparsed_count:
                    reparsed_miss_lemma_count += (org_count - reparsed_count)
                elif org_count < reparsed_count:
                    org_miss_lemma_count += (reparsed_count - org_count)

        # Whole file statistics:
        if mismatch_sent != 0:
            report_mismatches[corp_no][selected_lemma].append(f"Total mismatched sentences: {mismatch_sent} ({mismatch_sent/org_sent_count*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in original file (compared to org): {org_miss_lemma_count} ({org_miss_lemma_count/ (org_lemma_count)*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in reparsed file (compared to org): {reparsed_miss_lemma_count} ({reparsed_miss_lemma_count/org_lemma_count*100:.2f}%)")

In [ ]:
report_mismatches

In [ ]:
with open('./SemEval_lat/mismatch_report.txt', 'w') as f:
    # Write the report_mismatches dictionary to the file beautifully
    for corp_no in report_mismatches:
        f.write(f"Corpus {corp_no}:\n")
        for selected_lemma in report_mismatches[corp_no]:
            f.write(f"Lemma: {selected_lemma}\n")
            for mismatch in report_mismatches[corp_no][selected_lemma]:
                f.write(f"{mismatch}\n")
            f.write("\n")

## Fix the mismatches

In [ ]:
file_paths = [
    './SemEval_lat/corpus1/reparsed/lat1_reparsed.txt',
    './SemEval_lat/corpus2/reparsed/lat2_reparsed.txt'  
]

org_paths = [
    './semeval2020_ulscd_lat/corpus1/lemma/lat1.txt',
    './semeval2020_ulscd_lat/corpus2/lemma/lat2.txt'
]

### Noun vs PROPN

In [ ]:
N_PROPN_patterns = [
    r'\tancilla\tPROPN',
    r'\tcivitas\tPROPN',
    r'\tcohors\tPROPN',
    r'\tconsilium\tPROPN',
    r'\tconsul\tPROPN',
    r'\tdolus\tPROPN',
    r'\tdux\tPROPN',
    r'\thonor\tPROPN',
    r'\thostis\tPROPN',
    r'\thumanitas\tPROPN',
    r'\timperator\tPROPN',
    r'\tjus\tPROPN',
    r'\tnepos\tPROPN',
    r'\tnobilitas\tPROPN',
    r'\tpoena\tPROPN',
    r'\tpontifex\tPROPN',
    r'\tpotestas\tPROPN',
    r'\tregnum\tPROPN',
    r'\tsacramentum\tPROPN',
    r'\tsalus\tPROPN',
    r'\tsapientia\tPROPN',
    r'\tscriptura\tPROPN',
    r'\tsenatus\tPROPN',
    r'\tsensus\tPROPN',
    r'\ttemplum\tPROPN',
    r'\ttitulus\tPROPN',
    r'\tvirtus\tPROPN',
    r'\tvoluntas\tPROPN'
    ]

for file_path in file_paths:
    content = open_txt(file_path)
    for pattern in N_PROPN_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
N_PROPN_patterns_replacement ={
    r'\tconsilium\tPROPN': r'\tconsilium\tNOUN',
    r'\thonor\tPROPN': r'\thonor\tNOUN',
    r'\timperator\tPROPN': r'\timperator\tNOUN',
    r'\tjus\tPROPN': r'\tjus\tNOUN',
    r'\tpoena\tPROPN': r'\tpoena\tNOUN',
    r'\tpontifex\tPROPN': r'\tpontifex\tNOUN',
    r'\tsapientia\tPROPN': r'\tsapientia\tNOUN',
    r'\tsalus\tPROPN': r'\tsalus\tNOUN',
    r'\tsenatus\tPROPN': r'\tsenatus\tNOUN',
    r'\ttitulus\tPROPN': r'\ttitulus\tNOUN',
    r'\tnepos\tPROPN': r'\tnepos\tNOUN',
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in N_PROPN_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

### Spelling

In [ ]:
spelling_patterns = [
    r'\tacerba\tNOUN', # r'\tacerbus\tADJ',
    r'\tacerius\t', # r'\tacerbus\t',
    r'\tacerbus\tADV', # r'\tacerbus\tADJ',
    r'acerbus\tacer\tNOUN', # r'acerbus\tacerbus\tADJ',

    r'\tassumo\t', # r'\tadsumo\t',
    r'(adsump\w*)\tadsum\t', # r'\1\tadsumo\t',

    r'\tancillus\t', # r'\tancilla\t',

    r'\tciuitas\t', # r'\tcivitas\t',

    r'\tcohortus\t', # r'\tcohors\t',
    r'\tcohor\t', # r'\tcohors\t',

    r'(consul\w*)\tconsus\t', # r'\1\tconsul\t',
    
    r'\tdolum\t', # r'\tdolus\t',

    r'\tdubii\t', # r'\tdubius\t',
    r'\tdubi\w*\tADJ', # r'\tdubius\tADJ',

    r'^(hostis|hostem|hoste|hostes|hosti|hostibus|hostium)\t[^\t]+\tNOUN', # r'\1\thostis\tNOUN',

    r'\titeo\t', # r'\titero\t',

    r'\tius\t', # r'\tjus\t',
    
    r'^(nepos|nepotem|nepotis|nepoti|nepote|nepotes|nepotum|nepotibus)\t[^\t]+\tNOUN', # r'\1\tnepos\tNOUN'

    r'\tsenatum\tNOUN', # r'\tsenatus\tNOUN',

    r'\ttitula\tNOUN', # r'\ttitulus\tNOUN',

    r'\tuoluntas\t', # r'\tvoluntas\t'

    ]

for file_path in file_paths:
    print('CORPUS')
    content = open_txt(file_path)
    for pattern in spelling_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
spelling_patterns_replacement ={
    r'\tacerba\tNOUN': r'\tacerbus\tADJ',
    r'\tacerius\t': r'\tacerbus\t',
    r'\tacerbus\tADV': r'\tacerbus\tADJ',
    r'acerbus\tacer\tNOUN': r'acerbus\tacerbus\tADJ',

    r'\tassumo\t': r'\tadsumo\t',
    r'(adsump\w*)\tadsum\t': r'\1\tadsumo\t',

    r'\tancillus\t': r'\tancilla\t',

    r'\tciuitas\t': r'\tcivitas\t',

    r'\tcohortus\t': r'\tcohors\t',
    r'\tcohor\t': r'\tcohors\t',

    r'(consul\w*)\tconsus\t': r'\1\tconsul\t',
    
    r'\tdolum\t': r'\tdolus\t',

    r'\tdubii\t': r'\tdubius\t',
    r'\tdubi\w*\tADJ': r'\tdubius\tADJ',

    r'^(hostis|hostem|hoste|hostes|hosti|hostibus|hostium)\t[^\t]+\tNOUN': r'\1\thostis\tNOUN',

    r'\titeo\t': r'\titero\t',

    r'\tius\t': r'\tjus\t',
    
    r'^(nepos|nepotem|nepotis|nepoti|nepote|nepotes|nepotum|nepotibus)\t[^\t]+\tNOUN': r'\1\tnepos\tNOUN',

    r'\tsenatum\tNOUN': r'\tsenatus\tNOUN',

    r'\ttitula\tNOUN': r'\ttitulus\tNOUN',

    r'\tuoluntas\t': r'\tvoluntas\t'
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in spelling_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

## Merge 2 corpus

In [ ]:
merge_output =  pilot_folder / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)
corpus_1 = pilot_folder / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'lat(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'lat1_reparsed.txt'
file2_path = corpus_2 / f'lat2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)

## For SemEval: Unified the POS of the target

In [ ]:
SemEval_file_paths = [
    './SemEval_lat_SemEval/corpus1/reparsed/lat1_reparsed.txt',
    './SemEval_lat_SemEval/corpus2/reparsed/lat2_reparsed.txt'  
]

### N vs V vs A

In [ ]:
# NVA_SemEval_patterns = [
#     r'\tacerbe\t', # r'\tacerbus\t',

#     r'\tbeate\t', # r'\tbeatus\t',

#     r'\tfideliter\t', # r'\tfidelis\t',

#     r'\tlicet\tSCONJ', # r'\tlicet\tVERB',

#     r'\tnecessario\tADV', # r'\tnecessarius\tADJ',

#     r'\tsancte\t', # r'\tsanctus\t',

#     r'\tsimpliciter\tADV\t', # r'\tsimplex\tADJ\t'
# ]

# Replacements
NVA_SemEval_patterns_replacement ={
    r'\tacerbe\t': r'\tacerbus\t',

    r'\tbeate\t': r'\tbeatus\t',

    r'\tfideliter\t': r'\tfidelis\t',

    r'\tlicet\tSCONJ': r'\tlicet\tVERB',

    r'\tnecessario\tADV': r'\tnecessarius\tADJ',

    r'\tsancte\t': r'\tsanctus\t',

    r'\tsimpliciter\tADV\t': r'\tsimplex\tADJ\t'
    
    }

for file_path in SemEval_file_paths:
    content = open_txt(file_path)
    for org, repl in NVA_SemEval_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

### Unify POS

In [ ]:
SemEval_lemmas = [
    'acerbus',
    'adsumo',
    'ancilla',
    'beatus',
    'civitas',
    'cohors',
    'consilium',
    'consul',
    'credo',
    'dolus',
    'dubius',
    'dux',
    'fidelis',
    'honor',
    'hostis',
    'humanitas',
    'imperator',
    'itero',
    'jus',
    'licet',
    'necessarius',
    'nepos',
    'nobilitas',
    'oportet',
    'poena',
    'pontifex',
    'potestas',
    'regnum',
    'sacramentum',
    'salus',
    'sanctus',
    'sapientia',
    'scriptura',
    'senatus',
    'sensus',
    'simplex',
    'templum',
    'titulus',
    'virtus',
    'voluntas'
    ]

In [ ]:
import re

for file_path in SemEval_file_paths:
    content = open_txt(file_path)

    for lemma in SemEval_lemmas:
        # Match: \tlemma\t(ANY_POS)\t and replace the POS by TAR
        content = re.sub(
            rf'\t{re.escape(lemma)}\t[^\t]+\t',   # any POS between tabs
            f'\t{lemma}\tTAR\t',
            content
        )

    save_to_txt(content, file_path)


## Merge 2 corpus SemEval

In [ ]:
merge_output =  pilot_folder_SemEval / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)
corpus_1 = pilot_folder_SemEval / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder_SemEval / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'lat(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'lat1_reparsed.txt'
file2_path = corpus_2 / f'lat2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)